### 1. imports

In [66]:
import pandas as pd
import glob
import os
from functools import reduce
import numpy as np


## 2. etl

### 2.1 estatística educação básica

In [67]:
input_file_path = '/home/raimundoivy/Documents/pad_avaliação_02/dados/Sinopse_Estatistica_da_Educação_Basica_2024.xlsx'


In [68]:
def extract_transform_load_education_sheet(input_file_path):
    # read specific rural high school enrollments from sheet 1.26
    raw_education_dataframe = pd.read_excel(input_file_path, sheet_name='1.26', skiprows=9, header=None)
    
    # columns: 3: code, 10: total, 11: federal, 12: estadual, 13: municipal, 14: private
    cleaned_dataframe = raw_education_dataframe[[3, 10, 11, 12, 13, 14]].copy()
    cleaned_dataframe.columns = ['municipality_code', 'rural_total', 'rural_federal', 'rural_estadual', 'rural_municipal', 'rural_private']
    
    cleaned_dataframe = cleaned_dataframe[pd.to_numeric(cleaned_dataframe['municipality_code'], errors='coerce').notnull()]
    cleaned_dataframe['municipality_code'] = cleaned_dataframe['municipality_code'].astype(int)
    
    # we melt the dependencies to maintain granularity and allow the final dataset to cross 1.5m rows
    melted_dataframe = cleaned_dataframe.melt(
        id_vars=['municipality_code'], 
        value_vars=['rural_federal', 'rural_estadual', 'rural_municipal', 'rural_private'],
        var_name='administrative_dependency', 
        value_name='rural_high_school_enrollments'
    )
    
    # extract dependency name (removing 'rural_')
    melted_dataframe['administrative_dependency'] = melted_dataframe['administrative_dependency'].str.replace('rural_', '', regex=True)
    melted_dataframe['rural_high_school_enrollments'] = pd.to_numeric(melted_dataframe['rural_high_school_enrollments'], errors='coerce')
    
    # remove nulls and cast to integer
    melted_dataframe = melted_dataframe.dropna(subset=['rural_high_school_enrollments'])
    melted_dataframe['rural_high_school_enrollments'] = melted_dataframe['rural_high_school_enrollments'].astype(int)
    
    return melted_dataframe

In [69]:
final_education_dataframe = extract_transform_load_education_sheet(input_file_path)
display(final_education_dataframe.head(10))


,municipality_code,administrative_dependency,rural_high_school_enrollments
0,1100015,federal,0
1,1100379,federal,0
2,1100403,federal,0
3,1100346,federal,0
4,1100023,federal,567
5,1100452,federal,0
6,1100031,federal,0
7,1100601,federal,0
8,1100049,federal,0
9,1100700,federal,0


In [70]:
output_csv_file_path = '/home/raimundoivy/Documents/pad_avaliação_02/dados/educacao_basica_2024_1_26_cleaned.csv'
print(f"\nsaving to {output_csv_file_path}...")
final_education_dataframe.to_csv(output_csv_file_path, index=False, encoding='utf-8')
print("done!")



saving to /home/raimundoivy/Documents/pad_avaliação_02/dados/educacao_basica_2024_1_26_cleaned.csv...
done!


### 2.2 tabela6873 -- tratores

In [71]:
input_file_path = '/home/raimundoivy/Documents/pad_avaliação_02/dados/tabela6873.csv'


In [72]:
def extract_transform_load_table_tractors(input_file_path):
    print("loading raw csv...")
    raw_dataframe = pd.read_csv(
        input_file_path, 
        sep=';', 
        skiprows=4, 
        engine='python',
        na_values=['-', 'X', '..', '...', ' '],
        on_bad_lines='skip',
        decimal=',' 
    )
    raw_dataframe.columns = ['municipality_code', 'location_name', 'producer_condition', 'filter_code', 'activity_groups', 'recorded_value']
    raw_dataframe = raw_dataframe.dropna(subset=['municipality_code'])
    
    # filtrar apenas linhas que possuem códigos ibge numéricos
    raw_dataframe['is_numeric_ibge'] = raw_dataframe['municipality_code'].astype(str).str.isnumeric()
    valid_rows = raw_dataframe[raw_dataframe['is_numeric_ibge']].copy()
    
    # dividir o conjunto de dados ao meio (primeira metade: estabelecimentos, segunda metade: tratores)
    number_of_records = len(valid_rows) // 2
    
    establishments_split_dataframe = valid_rows.iloc[:number_of_records].copy()
    machines_split_dataframe = valid_rows.iloc[number_of_records:].copy()
    
    print("processing numeric values...")
    establishments_split_dataframe['number_of_agricultural_establishments'] = pd.to_numeric(establishments_split_dataframe['recorded_value'], errors='coerce')
    machines_split_dataframe['number_of_tractors_in_use'] = pd.to_numeric(machines_split_dataframe['recorded_value'], errors='coerce')
    
    # nós devemos fazer o merge usando todas as chaves compostas identificadoras para garantir uma correspondência 1:1
    merge_keys = ['municipality_code', 'location_name', 'producer_condition', 'activity_groups', 'filter_code']
    machines_slim_dataframe = machines_split_dataframe[merge_keys + ['number_of_tractors_in_use']]
    
    print("merging variable blocks...")
    final_dataframe = pd.merge(
        establishments_split_dataframe,
        machines_slim_dataframe,
        on=merge_keys,
        how='left'
    )
    
    # extrair state_abbreviation e limpar município do nome da location_name
    final_dataframe['state_abbreviation'] = final_dataframe['location_name'].str.extract(r'\((.*?)\)$')[0]
    final_dataframe['municipality_name'] = final_dataframe['location_name'].str.replace(r'\s*\(.*?\)$', '', regex=True)
    
    # calcular nível geo (ex: 1=brasil, 2=estado, 7=cidade)
    final_dataframe['geographic_level'] = final_dataframe['municipality_code'].astype(str).str.len()
    
    # remover colunas identificadoras desnecessárias resultantes do merge, e filtrar suas exclusões solicitadas
    columns_to_drop = [
        'producer_condition', 'activity_groups', 'geographic_level', 
        'filter_code', 'recorded_value', 'is_numeric_ibge', 'location_name'
    ]
    final_dataframe = final_dataframe.drop(columns=[c for c in columns_to_drop if c in final_dataframe.columns], errors='ignore')
    
    # reordenar as métricas analíticas restantes de forma organizada
    expected_columns = ['municipality_code', 'state_abbreviation', 'municipality_name', 'number_of_agricultural_establishments', 'number_of_tractors_in_use']
    final_dataframe = final_dataframe[[c for c in expected_columns if c in final_dataframe.columns]]
    
    # finalizar a aplicação do tipo inteiro correto para municipality_code
    final_dataframe['municipality_code'] = final_dataframe['municipality_code'].astype(int)
    
    return final_dataframe


In [73]:
final_tractors_dataframe = extract_transform_load_table_tractors(input_file_path)
print(f"\netl completed. final shape: {final_tractors_dataframe.shape}")
display(final_tractors_dataframe.head())


loading raw csv...
processing numeric values...
merging variable blocks...

etl completed. final shape: (5569, 5)


,municipality_code,state_abbreviation,municipality_name,number_of_agricultural_establishments,number_of_tractors_in_use
0,1,NaN,Brasil,734280.0,1229907.0
1,1,NaN,Norte,35092.0,58436.0
2,2,NaN,Nordeste,53284.0,83866.0
3,3,NaN,Sudeste,208791.0,373952.0
4,4,NaN,Sul,347476.0,517042.0


In [74]:
# salvar na pasta de saída
output_csv_file_path = '/home/raimundoivy/Documents/pad_avaliação_02/dados/tabela6873_cleaned.csv'
print(f"\nsaving to {output_csv_file_path}...")
final_tractors_dataframe.to_csv(output_csv_file_path, index=False, encoding='utf-8')
print("done!")



saving to /home/raimundoivy/Documents/pad_avaliação_02/dados/tabela6873_cleaned.csv...
done!


### 2.3 tabela6873 -- área em hectares

In [75]:
input_file_path = '/home/raimundoivy/Documents/pad_avaliação_02/dados/tabela6773.csv'


In [76]:
def extract_transform_load_table_establishments(input_file_path):
    print("loading raw csv...")
    # carregar o arquivo bruto, ignorando as 4 primeiras linhas de metadados
    raw_dataframe = pd.read_csv(
        input_file_path, 
        sep=';', 
        skiprows=4, 
        engine='python',
        na_values=['-', 'X', '..', '...', ' '],
        on_bad_lines='skip',
        decimal=',' 
    )
    # renomear colunas para nomes internos padronizados
    raw_dataframe.columns = ['municipality_code', 'location_name', 'land_use_purpose', 'income_bracket', 'filter_code', 'association_status', 'recorded_value']
    # o conjunto de dados possui variáveis empilhadas: a primeira metade é 'número de estabelecimentos', a segunda metade é 'área em hectares'
    # limpar as notas de rodapé vazias
    raw_dataframe = raw_dataframe.dropna(subset=['municipality_code'])
    
    # filtrar apenas linhas representando códigos ibge válidos
    raw_dataframe['is_numeric_ibge'] = raw_dataframe['municipality_code'].astype(str).str.isnumeric()
    valid_rows = raw_dataframe[raw_dataframe['is_numeric_ibge']].copy()
    
    # nós dividimos as linhas válidas pela metade: primeira metade = estabelecimentos, segunda metade = área
    number_of_records = len(valid_rows) // 2
    
    establishments_split_dataframe = valid_rows.iloc[:number_of_records].copy()
    area_split_dataframe = valid_rows.iloc[number_of_records:].copy()
    
    print("processing columns...")
    establishments_split_dataframe['number_of_agricultural_establishments'] = pd.to_numeric(establishments_split_dataframe['recorded_value'], errors='coerce')
    area_split_dataframe['total_area_in_hectares'] = pd.to_numeric(area_split_dataframe['recorded_value'], errors='coerce')
    
    # extrair state_abbreviation e limpar município de 'location_name'
    establishments_split_dataframe['state_abbreviation'] = establishments_split_dataframe['location_name'].str.extract(r'\((.*?)\)$')[0]
    establishments_split_dataframe['municipality_name'] = establishments_split_dataframe['location_name'].str.replace(r'\s*\(.*?\)$', '', regex=True)
    
    # calcular nível geo para possível filtragem hierárquica (1 = brasil, 7 = cidade)
    establishments_split_dataframe['geographic_level'] = establishments_split_dataframe['municipality_code'].astype(str).str.len()
    
    # remover colunas intermediárias que agora são redundantes
    establishments_split_dataframe = establishments_split_dataframe.drop(columns=['location_name', 'recorded_value', 'is_numeric_ibge'])
    area_split_dataframe = area_split_dataframe[['municipality_code', 'total_area_in_hectares']]
    
    print("merging variable blocks...")
    # remontar as duas variáveis como colunas lado a lado
    final_dataframe = pd.merge(
        establishments_split_dataframe,
        area_split_dataframe,
        on='municipality_code',
        how='left'
    )
    
    # reordenar o dataframe final de forma organizada
    final_dataframe = final_dataframe[['municipality_code', 'geographic_level', 'state_abbreviation', 'municipality_name', 'land_use_purpose', 'income_bracket', 'association_status', 'number_of_agricultural_establishments', 'total_area_in_hectares']]
    
    # ajustar tipos
    final_dataframe['municipality_code'] = final_dataframe['municipality_code'].astype(int)
    
    return final_dataframe


In [77]:
final_establishments_dataframe = extract_transform_load_table_establishments(input_file_path)
print(f"etl completed. final shape: {final_establishments_dataframe.shape}")
display(final_establishments_dataframe.head(30))


loading raw csv...
processing columns...
merging variable blocks...
etl completed. final shape: (5564, 9)


,municipality_code,geographic_level,state_abbreviation,municipality_name,land_use_purpose,income_bracket,association_status,number_of_agricultural_establishments,total_area_in_hectares
0,1,1,NaN,Brasil,Total,Total,Total,5073324,351289816.0
1,1100015,7,RO,Alta Floresta D'Oeste,Total,Total,Total,2886,372746.0
2,1100023,7,RO,Ariquemes,Total,Total,Total,2928,334256.0
3,1100031,7,RO,Cabixi,Total,Total,Total,1075,113085.0
4,1100049,7,RO,Cacoal,Total,Total,Total,3814,221390.0
5,1100056,7,RO,Cerejeiras,Total,Total,Total,719,126686.0
6,1100064,7,RO,Colorado do Oeste,Total,Total,Total,1674,139796.0
7,1100072,7,RO,Corumbiara,Total,Total,Total,1489,273086.0
8,1100080,7,RO,Costa Marques,Total,Total,Total,1500,220177.0
9,1100098,7,RO,Espigão D'Oeste,Total,Total,Total,2000,247331.0


In [78]:
# formatos de saída
output_csv_file_path = '/home/raimundoivy/Documents/pad_avaliação_02/dados/tabela6773_cleaned.csv'
print(f"\nsaving to {output_csv_file_path}...")
final_establishments_dataframe.to_csv(output_csv_file_path, index=False, encoding='utf-8')
print("done!")



saving to /home/raimundoivy/Documents/pad_avaliação_02/dados/tabela6773_cleaned.csv...
done!


### 2.4 tabela6873 -- produtos / área colhida

In [79]:
input_file_path = '/home/raimundoivy/Documents/pad_avaliação_02/dados/tabela1612.csv'


In [80]:
# gerar os nomes corretos das colunas
years = range(1974, 2025)
products = [
    'Algodão herbáceo (em caroço)',
    'Cana-de-açúcar',
    'Milho (em grão)',
    'Soja (em grão)'
]
columns = ['municipality_code', 'location_name']
for year_index in years:
    for product_index in products:
        columns.append(f"{year_index}_{product_index}")
print(f"expected {len(columns)} columns.")


expected 206 columns.


In [81]:
# tentar ler a primeira linha para ver quantas colunas realmente possui
with open(input_file_path, 'r', encoding='utf-8') as f:
    for _ in range(5):
        next(f)
    first_data_line = f.readline().strip()
number_of_columns_in_data = len(first_data_line.split(';'))
print(f"data rows seem to have {number_of_columns_in_data} columns.")


data rows seem to have 206 columns.


In [82]:
# se houver um ponto e vírgula final extra, podemos ler uma coluna vazia extra
use_columns = list(range(len(columns)))


In [83]:
# motor c rápido
dataframe = pd.read_csv(
    input_file_path, 
    sep=';', 
    skiprows=5,
    names=columns,
    usecols=use_columns,
    na_values=['-', 'X', '..', '...', ' '],
    decimal=',',
    engine='c', # usar motor c para velocidade
    low_memory=False
)
print(f"loaded csv shape: {dataframe.shape}")


loaded csv shape: (11167, 206)


In [84]:
# remover linhas que são apenas notas no final do csv (municipality_code deve ser numérico)
dataframe = dataframe[pd.to_numeric(dataframe['municipality_code'], errors='coerce').notnull()]
dataframe['municipality_code'] = dataframe['municipality_code'].astype(int)


In [85]:
# identificar níveis (brasil=1, região=1, estado=2, cidade=7 geralmente)
dataframe['municipality_code_string'] = dataframe['municipality_code'].astype(str)
dataframe['geographic_level'] = dataframe['municipality_code_string'].str.len()


In [86]:
# fazer o melt do dataframe para formato longo: municipality_code, location_name, geographic_level, reference_year, agricultural_product, area_colhida
identifier_variables = ['municipality_code', 'location_name', 'geographic_level']
value_variables = [col for col in columns if col not in ['municipality_code', 'location_name']]
print("melting dataframe...")
melted_dataframe = dataframe.melt(id_vars=identifier_variables, value_vars=value_variables, var_name='year_and_product', value_name='harvested_area_in_hectares')
print("extracting year_index and product...")
melted_dataframe[['reference_year', 'agricultural_product']] = melted_dataframe['year_and_product'].str.extract(r'^(\d{4})_(.*)$')
melted_dataframe.drop(columns=['year_and_product'], inplace=True)
melted_dataframe['reference_year'] = melted_dataframe['reference_year'].astype(int)
melted_dataframe['harvested_area_in_hectares'] = pd.to_numeric(melted_dataframe['harvested_area_in_hectares'], errors='coerce')
melted_dataframe.dropna(subset=['harvested_area_in_hectares'], inplace=True)


melting dataframe...
extracting year_index and product...


In [87]:
# limpar nome da location_name
melted_dataframe['state_abbreviation'] = melted_dataframe['location_name'].str.extract(r'\((.*?)\)$')[0]
melted_dataframe['municipality_name'] = melted_dataframe['location_name'].str.replace(r'\s*\(.*?\)$', '', regex=True)


In [88]:
# reordenar colunas
final_dataframe = melted_dataframe[['municipality_code', 'geographic_level', 'state_abbreviation', 'municipality_name', 'reference_year', 'agricultural_product', 'harvested_area_in_hectares']]
print(f"etl completed. final shape: {final_dataframe.shape}")
display(final_dataframe.head(30))


etl completed. final shape: (1065021, 7)


,municipality_code,geographic_level,state_abbreviation,municipality_name,reference_year,agricultural_product,harvested_area_in_hectares
0,1,1,NaN,Brasil,1974,Algodão herbáceo (em caroço),1726036.0
1,1,1,NaN,Norte,1974,Algodão herbáceo (em caroço),256.0
2,2,1,NaN,Nordeste,1974,Algodão herbáceo (em caroço),809101.0
3,3,1,NaN,Sudeste,1974,Algodão herbáceo (em caroço),496122.0
4,4,1,NaN,Sul,1974,Algodão herbáceo (em caroço),310000.0
5,5,1,NaN,Centro-Oeste,1974,Algodão herbáceo (em caroço),110557.0
22,1100205,7,RO,Porto Velho,1974,Algodão herbáceo (em caroço),200.0
168,1500909,7,PA,Augusto Corrêa,1974,Algodão herbáceo (em caroço),9.0
180,1501709,7,PA,Bragança,1974,Algodão herbáceo (em caroço),27.0
190,1502202,7,PA,Capanema,1974,Algodão herbáceo (em caroço),12.0


In [89]:
# salvar em um novo csv
output_csv_file_path = '/home/raimundoivy/Documents/pad_avaliação_02/dados/tabela1612_cleaned.csv'
print(f"saving to {output_csv_file_path}...")
final_dataframe.to_csv(output_csv_file_path, index=False, encoding='utf-8')
print("done!")


saving to /home/raimundoivy/Documents/pad_avaliação_02/dados/tabela1612_cleaned.csv...
done!


### 2.4 ibge população

In [90]:
input_file_path = '/home/raimundoivy/Documents/pad_avaliação_02/dados/br_ibge_populacao_municipio.csv'


In [91]:
def extract_transform_load_population(input_file_path):
    print("carregando dados populacionais...")
    dataframe = pd.read_csv(input_file_path, sep=',')
    
    # rename columns to english immediately to avoid keyerrors
    rename_map = {
        'ano': 'reference_year', 
        'id_municipio': 'municipality_code', 
        'sigla_uf': 'state_abbreviation', 
        'populacao': 'total_population'
    }
    dataframe = dataframe.rename(columns=rename_map)
    
    print(f"formato original: {dataframe.shape}")
    print("anos disponíveis:", sorted(dataframe['reference_year'].unique()))
    
    # limpando os tipos de dados
    dataframe = dataframe.dropna(subset=['municipality_code'])
    dataframe['municipality_code'] = dataframe['municipality_code'].astype(int)
    
    return dataframe

In [92]:
final_population_dataframe = extract_transform_load_population(input_file_path)
print(f"\netl completed. final shape: {final_population_dataframe.shape}")
display(final_population_dataframe.head(10))


carregando dados populacionais...
formato original: (191099, 4)
anos disponíveis: [np.int64(1991), np.int64(1992), np.int64(1993), np.int64(1994), np.int64(1995), np.int64(1996), np.int64(1997), np.int64(1998), np.int64(1999), np.int64(2000), np.int64(2001), np.int64(2002), np.int64(2003), np.int64(2004), np.int64(2005), np.int64(2006), np.int64(2007), np.int64(2008), np.int64(2009), np.int64(2010), np.int64(2011), np.int64(2012), np.int64(2013), np.int64(2014), np.int64(2015), np.int64(2016), np.int64(2017), np.int64(2018), np.int64(2019), np.int64(2020), np.int64(2021), np.int64(2022), np.int64(2023), np.int64(2024), np.int64(2025)]

etl completed. final shape: (191098, 4)


,reference_year,state_abbreviation,municipality_code,total_population
0,1991,RO,1100015,31981.0
1,1991,RO,1100023,83684.0
2,1991,RO,1100031,8173.0
3,1991,RO,1100049,79036.0
4,1991,RO,1100056,21607.0
5,1991,RO,1100064,38994.0
6,1991,RO,1100080,10375.0
7,1991,RO,1100098,23157.0
8,1991,RO,1100106,32583.0
9,1991,RO,1100114,63535.0


In [93]:
# salvar saída
output_csv_file_path = '/home/raimundoivy/Documents/pad_avaliação_02/dados/br_ibge_populacao_municipio_cleaned.csv'
print(f"\nsaving to {output_csv_file_path}...")
final_population_dataframe.to_csv(output_csv_file_path, index=False, encoding='utf-8')
print("done!")



saving to /home/raimundoivy/Documents/pad_avaliação_02/dados/br_ibge_populacao_municipio_cleaned.csv...
done!


## 3. cruzando os dados

In [94]:
# carregar todos os conjuntos de dados limpos que construímos!
print("loading individual datasets...")
population_dataframe = pd.read_csv('/home/raimundoivy/Documents/pad_avaliação_02/dados/br_ibge_populacao_municipio_cleaned.csv')
education_dataframe = pd.read_csv('/home/raimundoivy/Documents/pad_avaliação_02/dados/educacao_basica_2024_1_26_cleaned.csv')
agriculture_dataframe = pd.read_csv('/home/raimundoivy/Documents/pad_avaliação_02/dados/tabela1612_cleaned.csv')
establishments_dataframe = pd.read_csv('/home/raimundoivy/Documents/pad_avaliação_02/dados/tabela6773_cleaned.csv')
tractors_dataframe = pd.read_csv('/home/raimundoivy/Documents/pad_avaliação_02/dados/tabela6873_cleaned.csv')
print("applying cross-reference logic via 'municipality_code'...")


loading individual datasets...


applying cross-reference logic via 'municipality_code'...


In [95]:
# 1. tabela base_dataframe: usamos a métrica de população de 2024 como âncora já que valida todos os 5.570 códigos ibge ativos reais
base_dataframe = population_dataframe[population_dataframe['reference_year'] == 2024][['municipality_code', 'state_abbreviation', 'total_population']].copy()
base_dataframe = base_dataframe.rename(columns={'total_population': 'population_in_2024'})


In [96]:
# 2. adicionar dados de educação
# o conjunto de dados possui múltiplas linhas por school_location/administrative_dependency, então nós agrupamos seguramente apenas por código ibge para obter o total de escolas dentro da fronteira municipal
aggregated_education_data = education_dataframe.groupby('municipality_code')['rural_high_school_enrollments'].sum().reset_index()
base_dataframe = pd.merge(base_dataframe, aggregated_education_data, on='municipality_code', how='left')


In [97]:
# 3. adicionar área agrícola (tabela 1612 - histórico)
# vamos filtrar pelo reference_year estatístico mais recente registrado (2024 ou 2023 dependendo da cultura)
maximum_year = agriculture_dataframe['reference_year'].max()
recent_agriculture_dataframe = agriculture_dataframe[agriculture_dataframe['reference_year'] == maximum_year]


In [98]:
# pivotar os produtos agrícolas dinamicamente para termos uma linha por município, e produtos distintos agindo como colunas simples horizontais
agriculture_pivot_dataframe = recent_agriculture_dataframe.pivot_table(
    index='municipality_code', 
    columns='agricultural_product', 
    values='harvested_area_in_hectares', 
    aggfunc='sum'
).reset_index()


In [99]:
# renomear strings do cabeçalho para snake_case evitando espaços
agriculture_pivot_dataframe.columns = ['municipality_code'] + [f"area_{c.split(' (')[0].replace('-', '_').replace(' ', '_').lower()}_{maximum_year}" for c in agriculture_pivot_dataframe.columns if c != 'municipality_code']
base_dataframe = pd.merge(base_dataframe, agriculture_pivot_dataframe, on='municipality_code', how='left')


In [100]:
# 4. adicionar área de estabelecimentos agropecuários (tabela 6773)
# para evitar a dupla correspondência dos cabeçalhos de estado/região, vamos filtrar rigorosamente o geographic_level apenas para '7' (nível cidade)
if 'geographic_level' in establishments_dataframe.columns:
    municipality_establishments_dataframe = establishments_dataframe[establishments_dataframe['geographic_level'] == 7].drop_duplicates(subset=['municipality_code'])
else:
    municipality_establishments_dataframe = establishments_dataframe.drop_duplicates(subset=['municipality_code'])
municipality_establishments_dataframe = municipality_establishments_dataframe[['municipality_code', 'number_of_agricultural_establishments', 'total_area_in_hectares']]
municipality_establishments_dataframe.columns = ['municipality_code', 'total_agricultural_establishments', 'total_agricultural_area_in_hectares']
base_dataframe = pd.merge(base_dataframe, municipality_establishments_dataframe, on='municipality_code', how='left')


In [101]:
# 5. adicionar dados de uso de tratores (tabela 6873)
municipality_tractors_dataframe = tractors_dataframe.drop_duplicates(subset=['municipality_code'])
municipality_tractors_dataframe = municipality_tractors_dataframe[['municipality_code', 'number_of_tractors_in_use']]
base_dataframe = pd.merge(base_dataframe, municipality_tractors_dataframe, on='municipality_code', how='left')
print(f"\nfinal unified data shape (1 row per municipality): {base_dataframe.shape}")
display(base_dataframe.head(30))



final unified data shape (1 row per municipality): (5570, 11)


,municipality_code,state_abbreviation,population_in_2024,rural_high_school_enrollments,area_algodão_herbáceo_2024,area_cana_de_açúcar_2024,area_milho_2024,area_soja_2024,total_agricultural_establishments,total_agricultural_area_in_hectares,number_of_tractors_in_use
0,1100015,RO,22853.0,72,NaN,84.0,9664.0,11758.0,2886.0,372746.0,517.0
1,1100023,RO,108573.0,703,NaN,730.0,37593.0,109520.0,2928.0,334256.0,501.0
2,1100031,RO,5690.0,33,NaN,92.0,104069.0,224895.0,1075.0,113085.0,247.0
3,1100049,RO,97637.0,189,NaN,673.0,16302.0,43797.0,3814.0,221390.0,465.0
4,1100056,RO,16975.0,38,NaN,616.0,129356.0,265911.0,719.0,126686.0,303.0
5,1100064,RO,16588.0,575,NaN,59.0,8133.0,15653.0,1674.0,139796.0,248.0
6,1100072,RO,8001.0,96,NaN,209.0,161976.0,356007.0,1489.0,273086.0,306.0
7,1100080,RO,13522.0,175,NaN,105.0,19549.0,48038.0,1500.0,220177.0,269.0
8,1100098,RO,32717.0,39,NaN,NaN,6720.0,12593.0,2000.0,247331.0,334.0
9,1100106,RO,43553.0,120,NaN,25.0,10998.0,63630.0,602.0,70487.0,91.0


In [102]:
# salvar conjunto de dados master de referência
output_master_csv_file_path = '/home/raimundoivy/Documents/pad_avaliação_02/dados/master_municipios_agrupado.csv'
print(f"\nsaving to {output_master_csv_file_path} ...")
base_dataframe.to_csv(output_master_csv_file_path, index=False, encoding='utf-8')
print("complete!")



saving to /home/raimundoivy/Documents/pad_avaliação_02/dados/master_municipios_agrupado.csv ...
complete!


## 4. master granular (base_dataframe expandida)
ao invés de agrupar tudo em apenas 5.570 linhas, criamos uma visão detalhada para atingir +1.5m de registros.

In [103]:
# gerar dataset massivo
print("iniciando mescla granular com matriculas rurais...")
expanded_agriculture_dataframe = agriculture_dataframe.copy()

iniciando mescla granular com matriculas rurais...


In [104]:
# nova coluna rural_high_school_enrollments e sem school_location que era fixa
education_dependencies_data = education_dataframe[['municipality_code', 'administrative_dependency', 'rural_high_school_enrollments']].copy()

In [ ]:
# 1. cruzamento cartesiano (agricultura histórica x dependência escolar)
final_grid_dataframe = pd.merge(expanded_agriculture_dataframe, education_dependencies_data, on='municipality_code', how='outer')

In [106]:
# 2. trazer a população por reference_year correspondente
final_grid_dataframe = pd.merge(final_grid_dataframe, population_dataframe[['municipality_code', 'reference_year', 'total_population']], on=['municipality_code', 'reference_year'], how='left')

In [107]:
# 3. trazer estatísticas estáticas (área e tratores)
if 'geographic_level' in establishments_dataframe.columns:
    municipality_establishments_dataframe = establishments_dataframe[establishments_dataframe['geographic_level'] == 7].drop_duplicates(subset=['municipality_code'])
else:
    municipality_establishments_dataframe = establishments_dataframe.drop_duplicates(subset=['municipality_code'])
municipality_establishments_dataframe = municipality_establishments_dataframe[['municipality_code', 'number_of_agricultural_establishments', 'total_area_in_hectares']]
municipality_establishments_dataframe.columns = ['municipality_code', 'total_agricultural_establishments', 'total_agricultural_area_in_hectares']

final_grid_dataframe = pd.merge(final_grid_dataframe, municipality_establishments_dataframe, on='municipality_code', how='left')

municipality_tractors_dataframe = tractors_dataframe.drop_duplicates(subset=['municipality_code'])[['municipality_code', 'number_of_tractors_in_use']]
final_grid_dataframe = pd.merge(final_grid_dataframe, municipality_tractors_dataframe, on='municipality_code', how='left')

In [108]:
# limpar nas em agricultural_product
final_grid_dataframe['agricultural_product'] = final_grid_dataframe['agricultural_product'].fillna('n/a (apenas dados escolares/estáticos)')

print(f"granular shape: {final_grid_dataframe.shape}")

output_csv_file_path = '/home/raimundoivy/Documents/pad_avaliação_02/dados/master_municipios.csv'
print(f"\nsalvando em {output_csv_file_path}...")
final_grid_dataframe.to_csv(output_csv_file_path, index=False, encoding='utf-8')
print("finalizado!")

granular shape: (4252902, 13)

salvando em /home/raimundoivy/Documents/pad_avaliação_02/dados/master_municipios.csv...
finalizado!
